In [ ]:
import pandas as pd
import sqlite3

def load_csv_to_sql(csv_path, table_name="data", db_path=":memory:"):
    """Load a CSV into a SQLite table and return the connection."""
    df = pd.read_csv(csv_path)
    conn = sqlite3.connect(db_path)  # ":memory:" = temporary, in-RAM DB
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    return conn



def query(conn, sql):
    """Run a SQL query and return results as a DataFrame."""
    return pd.read_sql_query(sql, conn)

if __name__ == "__main__":
    conn = load_csv_to_sql("your_file.csv", table_name="data")

    result = query(conn, """
        SELECT *
        FROM data
        WHERE some_column > 100
        ORDER BY some_column DESC
        LIMIT 10
    """)

    print(result)
    conn.close()

In [ ]:
# import sqlite3

# def create_connection(db_path="my_database.db"):
#     """Create a connection to a SQLite database (creates file if it doesn't exist)."""
#     conn = sqlite3.connect(db_path)
#     return conn

# def create_table(conn):
#     """Create a table using raw SQL."""
#     create_table_sql = """
#     CREATE TABLE IF NOT EXISTS employees (
#         routeid INTEGER,
#         source CHAR(10),
#         destination CHAR(10),
#         base_fare REAL,
#         PRIMARY KEY (routeid, source, destination)
#         UNIQUE(source, destination),
#         FOREIGN KEY(source, destination) REFERENCES Flights(source, destination)
#     );
#     """
#     cursor = conn.cursor()
#     cursor.execute(create_table_sql)
#     conn.commit()
#     print("Table created (or already exists).")

# if __name__ == "__main__":
#     conn = create_connection("my_database.db")
#     create_table(conn)
#     conn.close()

Table created (or already exists).


In [2]:
import sqlite3
import pandas as pd

def create_connection(db_path="my_database.db"):
    return sqlite3.connect(db_path)

def create_table(conn):
    create_table_sql = """
    CREATE TABLE IF NOT EXISTS employees (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        department TEXT,
        salary REAL,
        hire_date DATE DEFAULT CURRENT_DATE
    );
    """
    conn.execute(create_table_sql)
    conn.commit()

def save_table_to_df(conn, table_name="employees"):
    """Read a table into a pandas DataFrame."""
    df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    return df

def load_df_to_table(conn, df, table_name="employees", if_exists="append"):
    """Write a DataFrame back into a table."""
    df.to_sql(table_name, conn, if_exists=if_exists, index=False)
    conn.commit()

if __name__ == "__main__":
    conn = create_connection("my_database.db")
    create_table(conn)

    # Pull table into a DataFrame
    df = save_table_to_df(conn, "employees")
    print(df)

    # (optional) save DataFrame to disk so you don't need to re-query
    df.to_csv("employees_backup.csv", index=False)
    # or: df.to_pickle("employees_backup.pkl")

    # ... later, reload the DataFrame back into the table ...
    load_df_to_table(conn, df, "employees", if_exists="append")

    # conn.close()


Empty DataFrame
Columns: [routeid, source, destination, base_fare]
Index: []


In [3]:
import sqlite3
import csv

def load_csv_to_table(conn, csv_path, table_name="flights"):
    cursor = conn.cursor()
    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        rows = [(row["flight_id"], row["source"], row["destination"], row["capacity"], row["seats_remaining"]) for row in reader]

    sql = f"INSERT INTO {table_name} (name, department, salary) VALUES (?, ?, ?)"
    cursor.executemany(sql, rows)
    conn.commit()
    print(f"Loaded {len(rows)} rows into {table_name}.")

if __name__ == "__main__":
    conn = sqlite3.connect("my_database.db")
    load_csv_to_table(conn, "flights.csv")
    conn.close()

OperationalError: no such table: flights

In [4]:
import sqlite3
import csv

def create_flights_table(conn):
    create_table_sql = """
    CREATE TABLE IF NOT EXISTS flights (
        flight_id TEXT PRIMARY KEY,
        source TEXT,
        destination TEXT,
        capacity INTEGER,
        seats_remaining INTEGER
    );
    """
    conn.execute(create_table_sql)
    conn.commit()

def load_csv_to_table(conn, csv_path, table_name="flights"):
    cursor = conn.cursor()
    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        rows = [
            (row["flight_id"], row["source"], row["destination"], row["capacity"], row["seats_remaining"])
            for row in reader
        ]

    sql = f"""
        INSERT INTO {table_name} (flight_id, source, destination, capacity, seats_remaining)
        VALUES (?, ?, ?, ?, ?)
    """
    cursor.executemany(sql, rows)
    conn.commit()
    print(f"Loaded {len(rows)} rows into {table_name}.")

if __name__ == "__main__":
    conn = sqlite3.connect("my_database.db")
    create_flights_table(conn)          # create the table first
    load_csv_to_table(conn, "flights.csv")
    conn.close()

Loaded 200 rows into flights.


In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("my_database.db")

def query(sql):
    """Helper: run a query and return a DataFrame."""
    return pd.read_sql_query(sql, conn)

# 1. View all flights
print(query("SELECT * FROM flights"))

# 2. Flights with available seats
print(query("""
    SELECT flight_id, source, destination, seats_remaining
    FROM flights
    WHERE seats_remaining > 0
"""))

# 3. Fully booked flights
print(query("""
    SELECT flight_id, source, destination
    FROM flights
    WHERE seats_remaining = 0
"""))

# 4. Flights

    flight_id     source destination  capacity  seats_remaining
0         PA0    Calgary   Vancouver       150                6
1         PA1    Toronto      Ottawa       175               35
2         PA2    Toronto      Ottawa       250              108
3         PA3  Vancouver     Toronto       150               55
4         PA4     Ottawa    Montreal       250               50
..        ...        ...         ...       ...              ...
195     PA195    Calgary   Vancouver       150              141
196     PA196    Toronto     Calgary       225               26
197     PA197   Montreal     Calgary       225              122
198     PA198    Calgary    Montreal       175               13
199     PA199   Montreal      Ottawa       150               16

[200 rows x 5 columns]
    flight_id     source destination  seats_remaining
0         PA0    Calgary   Vancouver                6
1         PA1    Toronto      Ottawa               35
2         PA2    Toronto      Ottawa          